# C8-embeddings — Practice p09 — Solution

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["guitar", "piano", "drums", "flute", "cello", "trumpet"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
S = W @ W.T


def top_k(S, i, k):
    order = np.argsort(S[i])[::-1]
    return order[order != i][:k]


top3_idx = top_k(S, 0, 3)
top3_words = np.asarray(WORDS)[top3_idx].tolist()
top3_sims = S[0, top3_idx]

Descending sorting ranks the row, and the index mask removes the query wherever it appears. For `guitar`, the three nearest candidates are `drums`, `piano`, and `trumpet`.

### Answer check

In [ ]:
assert W.shape == (6, 100) and W.dtype == np.float64
assert S.shape == (6, 6)
assert np.array_equal(top3_idx, np.array([2, 1, 5]))
assert top3_words == ["drums", "piano", "trumpet"]
assert np.allclose(top3_sims,
                   np.array([0.8045273217912854, 0.7777466020631806, 0.6862688633025621]),
                   atol=1e-12, rtol=0)